In [ ]:
import getpass
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

clave_pg = getpass.getpass("Contraseña de PostgreSQL: ")

url = URL.create(
    "postgresql+psycopg2",
    username="postgres",
    password=clave_pg,
    host="localhost",
    port=5432,
    database="tesis_agronomia",
)
engine = create_engine(url)

pd.read_sql("""
    SELECT cohorte, sede_ingreso, COUNT(*) AS estudiantes
    FROM tesis.estudiantes
    GROUP BY cohorte, sede_ingreso
    ORDER BY cohorte, sede_ingreso
""", engine)

In [ ]:
tablas = ["estudiantes", "actas_cursada", "examenes",
          "insc_cursada", "insc_examen", "materias", "nivelacion"]

datos = {t: pd.read_sql(f"SELECT * FROM tesis.{t}", engine) for t in tablas}

for t, df in datos.items():
    print(f"\n=== {t}: {len(df)} filas ===")
    resumen = pd.DataFrame({
        "tipo": df.dtypes.astype(str),
        "vacios": df.isna().sum(),
        "pct_vacios": (df.isna().mean() * 100).round(1),
        "valores_distintos": df.nunique(),
    })
    print(resumen.to_string())

In [ ]:
ac = datos["actas_cursada"]
ex = datos["examenes"]
ic = datos["insc_cursada"]
niv = datos["nivelacion"]
est = datos["estudiantes"]

print("1a. CURSADAS: notas vacías según origen y resultado")
print(ac.assign(tiene_nota=ac["nota"].notna())
        .groupby(["origen", "resultado", "tiene_nota"]).size().to_string())

print("\n1b. EXÁMENES: notas vacías según resultado")
print(ex.assign(tiene_nota=ex["nota"].notna())
        .groupby(["resultado", "tiene_nota"]).size().to_string())

print("\n2. INSCRIPCIONES A CURSADA por año académico")
print(ic["anio_academico"].value_counts().sort_index().to_string())

print("\n3. NIVELACIÓN")
print(niv.groupby(["anio_academico", "materia_nombre", "tipo", "resultado", "nota"],
                  dropna=False).size().to_string())
print("Estudiantes con nivelación por cohorte:")
print(est.merge(niv[["id_est"]].drop_duplicates(), on="id_est")
         ["cohorte"].value_counts().sort_index().to_string())

print("\n4. Sedes de las comisiones")
print(ac["ubicacion_comision"].value_counts().to_string())

In [ ]:
corte = pd.Timestamp("2026-03-31")

con_actividad = set(
    ac.loc[ac["anio_academico"] <= 2025, "id_est"]
).union(
    ic.loc[ic["anio_academico"] <= 2025, "id_est"]
).union(
    ex.loc[pd.to_datetime(ex["fecha"]) <= corte, "id_est"]
).union(
    datos["insc_examen"].loc[pd.to_datetime(datos["insc_examen"]["fecha"]) <= corte, "id_est"]
)

sin_actividad = est[~est["id_est"].isin(con_actividad)]
print("Estudiantes sin actividad hasta el año académico 2025:", len(sin_actividad))
print(sin_actividad[["cohorte", "sede_ingreso"]].value_counts().to_string())

In [ ]:
print("A. ¿Las promociones tienen también acta de regularidad?")
clave = ["id_est", "materia", "anio_academico", "periodo"]
origenes = ac.groupby(clave)["origen"].apply(lambda s: "".join(sorted(set(s))))
print(origenes.value_counts().to_string())

print("\nB. Valores de nota que aparecen")
print("Cursadas:", sorted(ac["nota"].dropna().unique()))
print("Exámenes:", sorted(ex["nota"].dropna().unique()))

print("\nC. Fechas de las actas fuera de su año académico")
f = pd.to_datetime(ac["fecha"])
fuera = ac[(f.dt.year < ac["anio_academico"]) | (f.dt.year > ac["anio_academico"] + 1)]
print("Cursadas con fecha fuera de rango:", len(fuera))

In [ ]:
# --- Detalle de las actas con fecha fuera de su año académico ---
print("Columnas disponibles:", list(ac.columns), "\n")

fuera = fuera.assign(anio_fecha=f[fuera.index].dt.year,
                     diferencia=f[fuera.index].dt.year - fuera["anio_academico"])
print("Diferencia entre el año de la fecha y el año académico:")
print(fuera["diferencia"].value_counts().sort_index().to_string(), "\n")

cols = [c for c in ["acta", "nro_acta", "materia", "anio_academico", "periodo",
                    "origen", "fecha", "anio_fecha", "resultado"] if c in fuera.columns]
print(fuera[cols].drop_duplicates().head(10).to_string(index=False))

In [ ]:
reg = ac[ac["origen"] == "R"].copy()
promocionadas = set(map(tuple, ac.loc[ac["origen"] == "P", clave].values))
reg["tiene_promocion"] = [tuple(x) in promocionadas for x in reg[clave].values]

print(reg[reg["nota"].isin(["A", "D", "N", "R", "U"])]
        .groupby(["nota", "resultado", "tiene_promocion"]).size().to_string())

In [ ]:
print(ex[ex["nota"] == "-"].groupby(["resultado", "tipo_acta"]).size().to_string())